In [ ]:
# Mount Drive & install dependencies
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix.git
!pip install -q visdom dominate

import sys
sys.path.insert(0, '/content/drive/MyDrive/Project2')

import torch
import torch.utils.data as data_utils
import os
from torchvision.utils import save_image
from utils.data_utils import get_mnist, get_usps, get_paired_loaders
from utils.fft_utils import fda_transfer
from utils.classifiers import LeNet5, train_classifier, evaluate_classifier
from utils.viz_utils import show_image_grid, print_results_table
from utils.eval_utils import save_results
from utils.cyclegan_wrapper import get_cyclegan_train_cmd, get_cyclegan_test_cmd
from utils.spectral_cyclegan import save_low_freq_images, reconstruct_from_translated_low
from utils.cycada import train_cycada, translate_with_cycada

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

SAVE_DIR = '/content/drive/MyDrive/Project2/results/mnist_usps'
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# === Quick Test Mode ===
# Set FAST_TEST = True to verify the notebook runs end-to-end (~5 min)
# Set FAST_TEST = False for full experiment
FAST_TEST = True

if FAST_TEST:
    CYCLEGAN_EPOCHS = 1
    CYCLEGAN_DECAY = 0
    CYCADA_EPOCHS = 1
    CLASSIFIER_EPOCHS = 2
    MAX_IMAGES = 100
    NUM_TEST = 50
else:
    CYCLEGAN_EPOCHS = 50
    CYCLEGAN_DECAY = 50
    CYCADA_EPOCHS = 50
    CLASSIFIER_EPOCHS = 20
    MAX_IMAGES = 5000
    NUM_TEST = 60000

print(f'FAST_TEST={FAST_TEST}: epochs={CYCLEGAN_EPOCHS}+{CYCLEGAN_DECAY}, images={MAX_IMAGES}')

In [ ]:
mnist_train = get_mnist(train=True, img_size=32)
mnist_test = get_mnist(train=False, img_size=32)
usps_train = get_usps(train=True, img_size=32)
usps_test = get_usps(train=False, img_size=32)

source_loader, target_loader = get_paired_loaders(mnist_train, usps_train, batch_size=64)
source_test_loader = torch.utils.data.DataLoader(mnist_test, batch_size=64, shuffle=False)
target_test_loader = torch.utils.data.DataLoader(usps_test, batch_size=64, shuffle=False)

print(f"MNIST train: {len(mnist_train)}, USPS train: {len(usps_train)}, USPS test: {len(usps_test)}")

# Task I: Style Transfer Comparison

In [ ]:
def save_dataset_images(dataset, output_dir, max_images=MAX_IMAGES):
    os.makedirs(output_dir, exist_ok=True)
    for i in range(min(len(dataset), max_images)):
        img, label = dataset[i]
        save_image(img, os.path.join(output_dir, f'{i:05d}.png'))
    print(f"Saved {min(len(dataset), max_images)} images to {output_dir}")

save_dataset_images(mnist_train, '/content/cyclegan_data/mnist2usps/trainA', max_images=MAX_IMAGES)
save_dataset_images(usps_train, '/content/cyclegan_data/mnist2usps/trainB', max_images=MAX_IMAGES)


In [ ]:
cmd = get_cyclegan_train_cmd(
    dataroot='/content/cyclegan_data/mnist2usps',
    name='mnist2usps_spatial',
    n_epochs=CYCLEGAN_EPOCHS, n_epochs_decay=CYCLEGAN_DECAY,
    load_size=32, crop_size=32,
    input_nc=1, output_nc=1,
    netG='resnet_6blocks'
)
print(f"Running: {cmd}")
!{cmd}

In [ ]:
# Decompose to low-freq
save_low_freq_images('/content/cyclegan_data/mnist2usps/trainA',
                     '/content/spectral_data/mnist', beta=0.03, mode='L')
save_low_freq_images('/content/cyclegan_data/mnist2usps/trainB',
                     '/content/spectral_data/usps', beta=0.03, mode='L')

# Prepare low-freq data for CycleGAN
import shutil
os.makedirs('/content/cyclegan_data/mnist2usps_spectral/trainA', exist_ok=True)
os.makedirs('/content/cyclegan_data/mnist2usps_spectral/trainB', exist_ok=True)
!cp /content/spectral_data/mnist/low/* /content/cyclegan_data/mnist2usps_spectral/trainA/
!cp /content/spectral_data/usps/low/* /content/cyclegan_data/mnist2usps_spectral/trainB/

cmd = get_cyclegan_train_cmd(
    dataroot='/content/cyclegan_data/mnist2usps_spectral',
    name='mnist2usps_spectral',
    n_epochs=CYCLEGAN_EPOCHS, n_epochs_decay=CYCLEGAN_DECAY,
    load_size=32, crop_size=32,
    input_nc=1, output_nc=1,
    netG='resnet_6blocks'
)
print(f"Running: {cmd}")
!{cmd}

In [ ]:
# Generate translated images
test_cmd_spatial = get_cyclegan_test_cmd(
    dataroot='/content/cyclegan_data/mnist2usps',
    name='mnist2usps_spatial', load_size=32, crop_size=32, input_nc=1, output_nc=1)
!{test_cmd_spatial}

test_cmd_spectral = get_cyclegan_test_cmd(
    dataroot='/content/cyclegan_data/mnist2usps_spectral',
    name='mnist2usps_spectral', load_size=32, crop_size=32, input_nc=1, output_nc=1)
!{test_cmd_spectral}

# Reconstruct spectral results
reconstruct_from_translated_low(
    './results/mnist2usps_spectral/test_latest/images',
    '/content/spectral_data/mnist/high',
    '/content/spectral_reconstructed/mnist2usps'
)

# Load and visualize
from torchvision import transforms
from PIL import Image

def load_images_from_dir(d, n=8):
    to_t = transforms.ToTensor()
    imgs = []
    for f in sorted(os.listdir(d)):
        if f.endswith('_fake_B.png'):
            imgs.append(to_t(Image.open(os.path.join(d, f)).convert('L')))
        if len(imgs) >= n:
            break
    return torch.stack(imgs) if imgs else None

original = torch.stack([mnist_train[i][0] for i in range(8)])
spatial = load_images_from_dir('./results/mnist2usps_spatial/test_latest/images', 8)

# Load spectral reconstructed images
recon_dir = '/content/spectral_reconstructed/mnist2usps'
spectral = None
if os.path.exists(recon_dir) and len(os.listdir(recon_dir)) > 0:
    to_t = transforms.ToTensor()
    spectral_imgs = []
    for f in sorted(os.listdir(recon_dir))[:8]:
        if f.lower().endswith(('.png', '.jpg')):
            spectral_imgs.append(to_t(Image.open(os.path.join(recon_dir, f)).convert('L')))
    if spectral_imgs:
        spectral = torch.stack(spectral_imgs)

viz_dict = {'Original MNIST': original}
if spatial is not None:
    viz_dict['Spatial CycleGAN → USPS'] = spatial
if spectral is not None:
    viz_dict['Spectral CycleGAN → USPS'] = spectral

show_image_grid(
    viz_dict,
    title='Task I: MNIST → USPS Style Transfer',
    save_path=os.path.join(SAVE_DIR, 'task1_comparison.png')
)

# Task II: UDA Benchmark

In [ ]:
results = {}
model_source = LeNet5(num_classes=10, in_channels=1)
model_source = train_classifier(model_source, source_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc = evaluate_classifier(model_source, target_test_loader, device=device)
results['Source-only'] = acc

In [ ]:
# Load CycleGAN-translated source images and pair with original labels
from torch.utils.data import TensorDataset, DataLoader

# Use CycleGAN to translate ALL source training images
# First, save test set for CycleGAN
save_dataset_images(mnist_train, '/content/cyclegan_data/mnist2usps_test/testA', max_images=min(NUM_TEST, len(mnist_train)))
save_dataset_images(usps_train, '/content/cyclegan_data/mnist2usps_test/testB', max_images=min(NUM_TEST, len(usps_train)))

test_all_cmd = get_cyclegan_test_cmd(
    dataroot='/content/cyclegan_data/mnist2usps_test',
    name='mnist2usps_spatial', load_size=32, crop_size=32, input_nc=1, output_nc=1)
!{test_all_cmd} --num_test {min(NUM_TEST, len(mnist_train))}

# Load translated images
from torchvision import transforms
from PIL import Image
to_tensor = transforms.ToTensor()

translated_imgs = []
result_dir = './results/mnist2usps_spatial/test_latest/images'
for f in sorted(os.listdir(result_dir)):
    if f.endswith('_fake_B.png'):
        img = to_tensor(Image.open(os.path.join(result_dir, f)).convert('L'))
        translated_imgs.append(img)

translated_imgs = torch.stack(translated_imgs[:len(mnist_train)])
labels = torch.tensor([mnist_train[i][1] for i in range(len(translated_imgs))])

cyclegan_dataset = TensorDataset(translated_imgs, labels)
cyclegan_loader = DataLoader(cyclegan_dataset, batch_size=64, shuffle=True)

model_cyclegan = LeNet5(num_classes=10, in_channels=1)
model_cyclegan = train_classifier(model_cyclegan, cyclegan_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc = evaluate_classifier(model_cyclegan, target_test_loader, device=device)
results['CycleGAN'] = acc

In [ ]:
# Similar to CycleGAN UDA but with spectral-translated images
save_dataset_images(mnist_train, '/content/cyclegan_data/mnist2usps_spectral_test/testA', max_images=min(NUM_TEST, len(mnist_train)))
save_dataset_images(usps_train, '/content/cyclegan_data/mnist2usps_spectral_test/testB', max_images=min(NUM_TEST, len(usps_train)))

test_spectral_cmd = get_cyclegan_test_cmd(
    dataroot='/content/cyclegan_data/mnist2usps_spectral_test',
    name='mnist2usps_spectral', load_size=32, crop_size=32, input_nc=1, output_nc=1)
!{test_spectral_cmd} --num_test {min(NUM_TEST, len(mnist_train))}

# Reconstruct: translated low-freq + original high-freq
# First decompose test images
save_low_freq_images('/content/cyclegan_data/mnist2usps_test/testA',
                     '/content/spectral_test_data/mnist', beta=0.03, mode='L')

reconstruct_from_translated_low(
    './results/mnist2usps_spectral/test_latest/images',
    '/content/spectral_test_data/mnist/high',
    '/content/spectral_reconstructed_all/mnist2usps'
)

# Load reconstructed images
spectral_imgs = []
recon_dir = '/content/spectral_reconstructed_all/mnist2usps'
for f in sorted(os.listdir(recon_dir)):
    if f.endswith('.png'):
        img = to_tensor(Image.open(os.path.join(recon_dir, f)).convert('L'))
        spectral_imgs.append(img)

spectral_imgs = torch.stack(spectral_imgs[:len(mnist_train)])
spectral_dataset = TensorDataset(spectral_imgs, labels[:len(spectral_imgs)])
spectral_loader = DataLoader(spectral_dataset, batch_size=64, shuffle=True)

model_spectral = LeNet5(num_classes=10, in_channels=1)
model_spectral = train_classifier(model_spectral, spectral_loader, epochs=CLASSIFIER_EPOCHS, lr=1e-3, device=device)
acc = evaluate_classifier(model_spectral, target_test_loader, device=device)
results['Spectral CycleGAN'] = acc

In [ ]:
G_s2t, G_t2s = train_cycada(
    source_loader, target_loader,
    classifier=model_source,
    num_classes=10, in_channels=1,
    n_epochs=CYCADA_EPOCHS, device=device
)

translated_data = translate_with_cycada(G_s2t, source_loader, device=device)
cycada_imgs = torch.cat([d[0] for d in translated_data])
cycada_labels = torch.cat([d[1] for d in translated_data])
cycada_dataset = TensorDataset(cycada_imgs, cycada_labels)
cycada_loader = DataLoader(cycada_dataset, batch_size=64, shuffle=True)

model_cycada = LeNet5(num_classes=10, in_channels=1)
model_cycada = train_classifier(model_cycada, cycada_loader, epochs=CLASSIFIER_EPOCHS, device=device)
acc = evaluate_classifier(model_cycada, target_test_loader, device=device)
results['CyCADA'] = acc

In [ ]:
fda_imgs = []
fda_labels = []

for (src_imgs, src_lbls), (tgt_imgs, _) in zip(source_loader, target_loader):
    bs = min(src_imgs.size(0), tgt_imgs.size(0))
    transferred = fda_transfer(src_imgs[:bs], tgt_imgs[:bs], beta=0.01)
    fda_imgs.append(transferred)
    fda_labels.append(src_lbls[:bs])

fda_imgs = torch.cat(fda_imgs)
fda_labels = torch.cat(fda_labels)

# Visualize FDA results
show_image_grid(
    {'Original MNIST': torch.stack([mnist_train[i][0] for i in range(8)]),
     'FDA Transferred': fda_imgs[:8]},
    title='FDA: MNIST \u2192 USPS',
    save_path=os.path.join(SAVE_DIR, 'fda_comparison.png')
)

fda_dataset = TensorDataset(fda_imgs, fda_labels)
fda_loader = DataLoader(fda_dataset, batch_size=64, shuffle=True)

model_fda = LeNet5(num_classes=10, in_channels=1)
model_fda = train_classifier(model_fda, fda_loader, epochs=CLASSIFIER_EPOCHS, device=device)
acc = evaluate_classifier(model_fda, target_test_loader, device=device)
results['FDA'] = acc

In [ ]:
print_results_table(results, 'MNIST \u2192 USPS')
save_results(results, 'MNIST_to_USPS', save_dir=SAVE_DIR)